In [ ]:
# ==============================================================================# 🚀 DO NOT MODIFY: Standardized Notebook Setup# ==============================================================================# This cell is designed to work in both Google Colab and local environments.# It ensures that the environment is correctly configured by cloning (or# locating) the project repository and installing the necessary dependencies.## ------------------------------------------------------------------------------##  ⚠️  IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY (NOT ON COLAB):##  This cell will automatically find the repository root and configure your#  environment. Just make sure you have run: pip install -e .[dev]## ------------------------------------------------------------------------------import osimport subprocessimport sysfrom pathlib import Path# --- Configuration ---REPO_URL = "https://github.com/BradSegal/ADH-LLM-Tutorials-2025.git"REPO_DIR = Path("ADH-LLM-Tutorials-2025")  # The name of the directory once cloned# --- End of Configuration ---def find_repo_root(start_path: Path) -> Path | None:    """    Find the repository root by looking for pyproject.toml.    Searches upward from start_path until it finds pyproject.toml or hits root.    Args:        start_path: Directory to start searching from.    Returns:        Path to repository root, or None if not found.    """    current = start_path.resolve()    while current \!= current.parent:  # Stop at filesystem root        if (current / "pyproject.toml").exists():            return current        current = current.parent    return Nonedef detect_active_branch(repo_dir: Path) -> str:    """    Determine the active git branch for pulling updates.    Tries multiple methods to detect the current branch name.    Args:        repo_dir: Path to the git repository.    Returns:        Branch name (defaults to 'master' if detection fails).    """    commands = [        "git symbolic-ref --short HEAD",        "git rev-parse --abbrev-ref HEAD",    ]    for cmd in commands:        result = subprocess.run(            cmd, shell=True, cwd=repo_dir, capture_output=True, text=True        )        if result.returncode == 0:            branch = result.stdout.strip()            if branch and not branch.startswith("origin/"):                return branch    return "master"def run_cmd(cmd: str, *, cwd: Path | None = None) -> None:    """    Run a shell command and raise an error if it fails.    Args:        cmd: The command to run.        cwd: Optional working directory for the command.    Raises:        RuntimeError: If the command returns a non-zero exit code.    """    result = subprocess.run(cmd, shell=True, cwd=cwd)    if result.returncode \!= 0:        raise RuntimeError(f"Command failed with exit code {result.returncode}: {cmd}")# --- Detect environment ---try:    import google.colab  # noqa: F401    IN_COLAB = Trueexcept ImportError:    IN_COLAB = False# --- Main setup logic ---if IN_COLAB:    print("☁️  Running in Google Colab. Setting up the environment...")    # Determine repository path    start_dir = Path.cwd()    if start_dir.name == REPO_DIR.name:        repo_path = start_dir    else:        repo_path = start_dir / REPO_DIR    # Clone or update repository    if not repo_path.exists():        print(f"📥 Cloning repository from {REPO_URL}...")        run_cmd(f"git clone --quiet {REPO_URL} {repo_path}")        print(f"✅ Repository cloned to {repo_path}")    else:        print(f"📂 Repository already exists at {repo_path}")        active_branch = detect_active_branch(repo_path)        print(f"🔄 Pulling latest changes from branch '{active_branch}'...")        run_cmd(f"git pull origin {active_branch} --quiet", cwd=repo_path)        print(f"✅ Repository updated")    # Verify repository structure    if not (repo_path / "pyproject.toml").exists():        raise FileNotFoundError(            f"Repository structure invalid: pyproject.toml not found in {repo_path}. "            "The repository may be corrupted."        )    # Change working directory and update Python path    print(f"📁 Changing working directory to {repo_path}")    os.chdir(repo_path)    if str(repo_path) not in sys.path:        sys.path.insert(0, str(repo_path))    # Install dependencies (smart installation - only installs missing packages)    from core.notebook.setup import smart_install_dependencies    result = smart_install_dependencies(        repo_path=repo_path,        include_dev=True,        verbose=True,    )    # Fail loudly if critical packages failed to install    if result["failed"]:        print(f"⚠️  WARNING: {len(result['failed'])} packages failed to install:")        for pkg in result["failed"]:            print(f"  - {pkg}")        print("You may encounter import errors. Please check your internet connection.")    print("" + "=" * 70)    print("✅ Environment setup complete\! You can now proceed with the notebook.")    print("=" * 70)else:    print("💻 Running in local environment. Configuring...")    # Find the repository root    repo_path = find_repo_root(Path.cwd())    if repo_path is None:        raise FileNotFoundError(            "Could not find repository root (no pyproject.toml found). "            "Please ensure you are running this notebook from within the "            "ADH-LLM-Tutorials-2025 repository directory."        )    print(f"✅ Found repository root: {repo_path}")    # Change working directory and update Python path    print(f"📁 Changing working directory to {repo_path}")    os.chdir(repo_path)    if str(repo_path) not in sys.path:        sys.path.insert(0, str(repo_path))    print("" + "=" * 70)    print("✅ Local environment configured successfully\!")    print("=" * 70)    print("⚠️  Please ensure you have run: pip install -e .[dev]")    print("   (Required for local development)")

# 08 - Practical Embeddings: Building a Semantic Search Engine

## From Concept to Application

In the previous notebook, you learned that embeddings convert text into numerical vectors that capture semantic meaning. Now we'll put this concept to work by building a **semantic search engine**.

### What is Semantic Search?

Unlike traditional keyword search (which looks for exact word matches), semantic search understands the **meaning** behind your query and finds documents that are conceptually similar, even if they use different words.

**Example:**
- Query: "What causes chest discomfort?"
- Traditional search: Looks for documents containing "chest" AND "discomfort"
- Semantic search: Finds documents about "angina," "heart attack," "cardiac pain," etc.

### Learning Objectives

By the end of this notebook, you will:
1. Create a small corpus of clinical FAQ documents
2. Embed the entire corpus into vector representations
3. Perform semantic search using cosine similarity
4. Understand the difference between semantic and keyword-based search

Let's build your first semantic search engine!

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import plotly.express as px
import umap
from scipy.spatial.distance import cdist

from core.llm import EmbeddingRequest, generate_embeddings

## Step 1: Create a Document Corpus

We'll create a knowledge base of clinical documents covering diverse medical domains. In a real-world application, this could be:
- Patient education materials
- Clinical practice guidelines
- Medical literature abstracts
- EHR documentation templates
- Drug information sheets
- Treatment protocols

For this tutorial, we'll use a structured corpus organized by medical domain, with each document containing clinically relevant information.

In [ ]:
# Create a comprehensive corpus of clinical documents organized by domain
corpus_data = [
    # Cardiovascular Domain
    {
        "id": 1,
        "category": "Cardiovascular",
        "topic": "Myocardial Infarction Symptoms",
        "content": (
            "Myocardial infarction (heart attack) symptoms include sudden onset chest pain or pressure, "
            "often described as crushing or squeezing. Pain may radiate to the left arm, jaw, neck, or back. "
            "Associated symptoms include shortness of breath, diaphoresis (sweating), nausea, vomiting, "
            "and a sense of impending doom. Atypical presentations are common in women, diabetics, and "
            "elderly patients, who may present with fatigue, dyspnea, or epigastric discomfort."
        ),
    },
    {
        "id": 2,
        "category": "Cardiovascular",
        "topic": "Hypertension Management Guidelines",
        "content": (
            "Hypertension management follows a stepwise approach. First-line therapy includes lifestyle "
            "modifications: sodium restriction, weight loss, regular aerobic exercise, and limiting alcohol. "
            "Pharmacologic treatment typically starts with thiazide diuretics, ACE inhibitors, ARBs, or "
            "calcium channel blockers. Target blood pressure is generally <130/80 mmHg. Resistant hypertension "
            "requires combination therapy and screening for secondary causes."
        ),
    },
    {
        "id": 3,
        "category": "Cardiovascular",
        "topic": "Atrial Fibrillation Overview",
        "content": (
            "Atrial fibrillation is the most common sustained cardiac arrhythmia, characterized by irregular, "
            "rapid atrial activation. Patients may experience palpitations, dyspnea, fatigue, or be asymptomatic. "
            "Management focuses on rate control (beta-blockers, calcium channel blockers), rhythm control "
            "(antiarrhythmic drugs, cardioversion), and anticoagulation to prevent thromboembolic stroke. "
            "CHA2DS2-VASc score guides anticoagulation decisions."
        ),
    },
    {
        "id": 4,
        "category": "Cardiovascular",
        "topic": "Heart Failure Treatment",
        "content": (
            "Chronic heart failure treatment aims to reduce symptoms and prevent disease progression. "
            "Cornerstone medications include ACE inhibitors or ARBs, beta-blockers, mineralocorticoid receptor "
            "antagonists, and SGLT2 inhibitors. Diuretics manage volume overload. Advanced therapies include "
            "cardiac resynchronization therapy, implantable cardioverter-defibrillators, and in severe cases, "
            "ventricular assist devices or heart transplantation."
        ),
    },
    {
        "id": 5,
        "category": "Cardiovascular",
        "topic": "Stroke Prevention and Recognition",
        "content": (
            "Stroke prevention requires managing modifiable risk factors: hypertension, diabetes, hyperlipidemia, "
            "atrial fibrillation, and smoking. The FAST mnemonic aids recognition: Face drooping, Arm weakness, "
            "Speech difficulty, Time to call emergency services. Acute ischemic stroke treatment with alteplase "
            "or mechanical thrombectomy is time-sensitive, with a narrow therapeutic window. Every minute matters."
        ),
    },
    {
        "id": 6,
        "category": "Cardiovascular",
        "topic": "Cardiac Catheterization Procedures",
        "content": (
            "Cardiac catheterization is an invasive diagnostic and therapeutic procedure. Coronary angiography "
            "visualizes coronary arteries to identify stenoses. Percutaneous coronary intervention (PCI) with "
            "angioplasty and stent placement can restore blood flow. Post-procedure care includes monitoring "
            "for bleeding, hematoma, or vascular complications. Dual antiplatelet therapy with aspirin and "
            "a P2Y12 inhibitor is essential to prevent stent thrombosis."
        ),
    },
    # Endocrine/Metabolic Domain
    {
        "id": 7,
        "category": "Endocrine",
        "topic": "Type 2 Diabetes Management",
        "content": (
            "Type 2 diabetes management emphasizes lifestyle modification as first-line therapy: medical "
            "nutrition therapy, regular physical activity, and weight loss. Metformin is the preferred "
            "initial pharmacologic agent unless contraindicated. Additional agents include SGLT2 inhibitors, "
            "GLP-1 receptor agonists, DPP-4 inhibitors, and insulin when needed. Target HbA1c is typically "
            "<7% for most patients, with individualized goals based on comorbidities and life expectancy."
        ),
    },
    {
        "id": 8,
        "category": "Endocrine",
        "topic": "Hypoglycemia Recognition",
        "content": (
            "Hypoglycemia occurs when blood glucose drops below 70 mg/dL. Symptoms progress from autonomic "
            "responses (tremor, palpitations, sweating, hunger) to neuroglycopenic symptoms (confusion, "
            "difficulty concentrating, seizures, loss of consciousness). Treatment follows the rule of 15: "
            "consume 15 grams of fast-acting carbohydrate, recheck glucose in 15 minutes, and repeat if needed. "
            "Severe hypoglycemia may require glucagon injection or intravenous dextrose."
        ),
    },
    {
        "id": 9,
        "category": "Endocrine",
        "topic": "Thyroid Dysfunction",
        "content": (
            "Hypothyroidism presents with fatigue, weight gain, cold intolerance, constipation, and bradycardia. "
            "Treatment is levothyroxine replacement with TSH monitoring. Hyperthyroidism causes weight loss, "
            "heat intolerance, tachycardia, and anxiety. Management options include antithyroid medications "
            "(methimazole, propylthiouracil), radioactive iodine ablation, or thyroidectomy. Both conditions "
            "require regular monitoring and dose adjustments."
        ),
    },
    {
        "id": 10,
        "category": "Endocrine",
        "topic": "Insulin Therapy Regimens",
        "content": (
            "Insulin therapy is essential for type 1 diabetes and advanced type 2 diabetes. Basal-bolus "
            "regimens combine long-acting insulin (glargine, detemir, degludec) for baseline coverage with "
            "rapid-acting insulin (lispro, aspart, glulisine) for mealtime glucose control. Insulin pumps "
            "provide continuous subcutaneous insulin infusion. Proper injection technique, rotation of sites, "
            "and carbohydrate counting are crucial for optimal glycemic control."
        ),
    },
    # Respiratory Domain
    {
        "id": 11,
        "category": "Respiratory",
        "topic": "Asthma Management",
        "content": (
            "Asthma is managed with a stepwise approach based on severity and control. Quick-relief medications "
            "include short-acting beta-agonists (albuterol) for acute symptoms. Controller medications prevent "
            "exacerbations: inhaled corticosteroids are first-line, with add-on therapies including long-acting "
            "beta-agonists, leukotriene modifiers, or biologics. Identifying and avoiding triggers, proper "
            "inhaler technique, and written action plans improve outcomes."
        ),
    },
    {
        "id": 12,
        "category": "Respiratory",
        "topic": "COPD Exacerbation",
        "content": (
            "Chronic obstructive pulmonary disease exacerbations present with increased dyspnea, cough, and "
            "sputum production. Treatment includes bronchodilators (short-acting beta-agonists and "
            "anticholinergics), systemic corticosteroids, and antibiotics if bacterial infection is suspected. "
            "Oxygen therapy maintains saturation 88-92%. Severe exacerbations may require non-invasive "
            "positive pressure ventilation or mechanical ventilation. Smoking cessation is paramount."
        ),
    },
    {
        "id": 13,
        "category": "Respiratory",
        "topic": "Pneumonia Diagnosis",
        "content": (
            "Community-acquired pneumonia diagnosis combines clinical presentation (fever, cough, dyspnea, "
            "pleuritic chest pain) with imaging and laboratory findings. Chest X-ray typically shows infiltrates. "
            "Severity assessment uses CURB-65 or PSI scores to guide disposition. Empiric antibiotics target "
            "common pathogens: macrolides or doxycycline for outpatients, beta-lactam plus macrolide or "
            "respiratory fluoroquinolone for hospitalized patients."
        ),
    },
    {
        "id": 14,
        "category": "Respiratory",
        "topic": "Pulmonary Embolism",
        "content": (
            "Pulmonary embolism presents with sudden-onset dyspnea, pleuritic chest pain, tachycardia, and "
            "hypoxemia. Risk stratification uses Wells criteria or PE rule-out criteria. D-dimer testing has "
            "high sensitivity in low-risk patients. CT pulmonary angiography is the gold standard for diagnosis. "
            "Treatment includes anticoagulation with heparin bridging to warfarin, DOACs, or thrombolysis for "
            "massive PE causing hemodynamic instability."
        ),
    },
    # Renal Domain
    {
        "id": 15,
        "category": "Renal",
        "topic": "Chronic Kidney Disease Stages",
        "content": (
            "Chronic kidney disease is classified into five stages based on estimated glomerular filtration rate. "
            "Stage 1 (GFR ≥90) and 2 (60-89) have normal or mildly reduced function with kidney damage markers. "
            "Stage 3 (30-59) shows moderate reduction. Stage 4 (15-29) is severe reduction. Stage 5 (<15) is "
            "kidney failure requiring dialysis or transplantation. Management focuses on treating underlying "
            "causes, blood pressure control, and avoiding nephrotoxins."
        ),
    },
    {
        "id": 16,
        "category": "Renal",
        "topic": "Acute Kidney Injury",
        "content": (
            "Acute kidney injury is characterized by rapid decline in renal function, marked by rising creatinine "
            "or decreased urine output. Causes include prerenal (hypovolemia, hypotension), intrinsic (acute "
            "tubular necrosis, glomerulonephritis), and postrenal (obstruction). Management addresses underlying "
            "cause, optimizes hemodynamics, avoids nephrotoxic drugs, and adjusts medication doses. Severe cases "
            "may require renal replacement therapy."
        ),
    },
    {
        "id": 17,
        "category": "Renal",
        "topic": "Urinary Tract Infections",
        "content": (
            "Urinary tract infections present with dysuria, frequency, urgency, and suprapubic pain. "
            "Pyelonephritis adds fever, flank pain, and systemic symptoms. Urinalysis shows pyuria and bacteriuria. "
            "Uncomplicated cystitis is treated with nitrofurantoin, trimethoprim-sulfamethoxazole, or fosfomycin. "
            "Pyelonephritis requires fluoroquinolones or ceftriaxone. Recurrent UTIs warrant investigation for "
            "anatomic abnormalities or resistant organisms."
        ),
    },
    # Neurological Domain
    {
        "id": 18,
        "category": "Neurological",
        "topic": "Seizure Disorders",
        "content": (
            "Epilepsy is characterized by recurrent unprovoked seizures. Generalized seizures include tonic-clonic, "
            "absence, and myoclonic types. Focal seizures originate from localized brain regions. Antiepileptic "
            "drugs are selected based on seizure type: valproic acid for generalized seizures, carbamazepine "
            "or lamotrigine for focal seizures. Refractory epilepsy may require polytherapy, vagal nerve "
            "stimulation, or surgical resection."
        ),
    },
    {
        "id": 19,
        "category": "Neurological",
        "topic": "Migraine Headaches",
        "content": (
            "Migraine is a neurovascular disorder causing recurrent severe headaches, often unilateral and "
            "pulsating, lasting 4-72 hours. Associated symptoms include photophobia, phonophobia, nausea, and "
            "vomiting. Aura may precede headache. Acute treatment includes NSAIDs, triptans, or antiemetics. "
            "Preventive therapy (beta-blockers, antiepileptics, CGRP antagonists) is considered for frequent "
            "or disabling migraines. Trigger identification and avoidance are important."
        ),
    },
    {
        "id": 20,
        "category": "Neurological",
        "topic": "Dementia Care",
        "content": (
            "Dementia involves progressive cognitive decline affecting daily functioning. Alzheimer's disease "
            "is most common, characterized by memory loss, executive dysfunction, and behavioral changes. "
            "Evaluation includes cognitive testing, neuroimaging, and laboratory studies to exclude reversible "
            "causes. Cholinesterase inhibitors (donepezil, rivastigmine) and memantine provide modest symptomatic "
            "benefit. Comprehensive care addresses safety, caregiver support, and advance care planning."
        ),
    },
    # Mental Health Domain
    {
        "id": 21,
        "category": "Mental Health",
        "topic": "Major Depression Treatment",
        "content": (
            "Major depressive disorder is characterized by persistent depressed mood or anhedonia for at least "
            "two weeks, with neurovegetative symptoms (sleep changes, appetite changes, fatigue, concentration "
            "difficulty). Treatment combines psychotherapy (cognitive-behavioral therapy, interpersonal therapy) "
            "with antidepressants (SSRIs, SNRIs as first-line). Severe or treatment-resistant depression may "
            "require ECT, TMS, or ketamine. Suicide risk assessment is essential."
        ),
    },
    {
        "id": 22,
        "category": "Mental Health",
        "topic": "Anxiety Disorders",
        "content": (
            "Anxiety disorders include generalized anxiety disorder, panic disorder, social anxiety, and specific "
            "phobias. Symptoms include excessive worry, restlessness, muscle tension, and autonomic arousal. "
            "Panic attacks cause sudden intense fear with palpitations, sweating, trembling, and sense of doom. "
            "Treatment includes CBT, exposure therapy, and medications (SSRIs, SNRIs, buspirone). Benzodiazepines "
            "provide short-term relief but carry dependence risk."
        ),
    },
    {
        "id": 23,
        "category": "Mental Health",
        "topic": "PTSD Management",
        "content": (
            "Post-traumatic stress disorder develops after exposure to trauma, with symptoms clustered into "
            "intrusion (flashbacks, nightmares), avoidance, negative mood/cognition alterations, and hyperarousal. "
            "Evidence-based treatments include trauma-focused CBT, prolonged exposure therapy, EMDR, and "
            "medications (SSRIs, SNRIs, prazosin for nightmares). Comorbid depression and substance use require "
            "integrated treatment approaches."
        ),
    },
    # Gastrointestinal Domain
    {
        "id": 24,
        "category": "Gastrointestinal",
        "topic": "GERD Management",
        "content": (
            "Gastroesophageal reflux disease presents with heartburn and regurgitation. Complications include "
            "esophagitis, strictures, and Barrett's esophagus. Lifestyle modifications include weight loss, "
            "head-of-bed elevation, avoiding late meals, and eliminating triggers. Pharmacotherapy uses proton "
            "pump inhibitors as first-line therapy, with H2-receptor antagonists for mild symptoms. Refractory "
            "GERD may require surgical fundoplication."
        ),
    },
    {
        "id": 25,
        "category": "Gastrointestinal",
        "topic": "Inflammatory Bowel Disease",
        "content": (
            "Inflammatory bowel disease encompasses Crohn's disease and ulcerative colitis. Crohn's can affect "
            "any GI tract segment with transmural inflammation, causing abdominal pain, diarrhea, and weight loss. "
            "Ulcerative colitis involves continuous colonic inflammation with bloody diarrhea. Treatment includes "
            "aminosalicylates, corticosteroids, immunomodulators (azathioprine, methotrexate), and biologics "
            "(anti-TNF agents, anti-integrin antibodies). Surgery may be necessary for complications."
        ),
    },
    # Medication Information
    {
        "id": 26,
        "category": "Medication",
        "topic": "Anticoagulation Therapy",
        "content": (
            "Anticoagulation prevents and treats thromboembolic disease. Warfarin requires INR monitoring with "
            "target 2-3 for most indications. Direct oral anticoagulants (DOACs) - apixaban, rivaroxaban, "
            "dabigatran, edoxaban - offer fixed dosing without monitoring but require dose adjustment for renal "
            "function. Bleeding risk assessment uses HAS-BLED score. Reversal agents include vitamin K and "
            "prothrombin complex concentrate for warfarin, idarucizumab for dabigatran."
        ),
    },
    {
        "id": 27,
        "category": "Medication",
        "topic": "Statin Therapy",
        "content": (
            "Statins are HMG-CoA reductase inhibitors that reduce cardiovascular risk by lowering LDL cholesterol. "
            "High-intensity statins (atorvastatin 40-80mg, rosuvastatin 20-40mg) achieve >50% LDL reduction. "
            "Indications include established ASCVD, LDL ≥190 mg/dL, diabetes age 40-75, and primary prevention "
            "based on 10-year ASCVD risk. Side effects include myalgias, elevated transaminases, and rarely "
            "rhabdomyolysis. Monitoring includes baseline and periodic lipid panels and liver enzymes."
        ),
    },
    {
        "id": 28,
        "category": "Medication",
        "topic": "Antibiotic Stewardship",
        "content": (
            "Antibiotic stewardship promotes appropriate antibiotic use to combat resistance, reduce adverse "
            "effects, and decrease costs. Principles include prescribing only when bacterial infection is likely, "
            "selecting narrow-spectrum agents, using appropriate dosing and duration, and de-escalating based on "
            "culture results. Common stewardship targets include avoiding antibiotics for viral URIs, limiting "
            "fluoroquinolone use, and optimizing surgical prophylaxis timing and duration."
        ),
    },
]

# Convert to DataFrame for easier manipulation
corpus_df = pd.DataFrame(corpus_data)

print(f"Corpus size: {len(corpus_df)} documents")
print(f"Unique categories: {len(corpus_df['category'].unique())}\n")

print("Category distribution:")
from collections import Counter

category_counts = Counter(corpus_df["category"])
for category, count in sorted(category_counts.items(), key=lambda x: -x[1]):
    print(f"  {category}: {count} documents")

print("\n" + "=" * 80)
print("Sample documents:")
print("=" * 80)
for i in [0, 6, 10, 14, 17, 20, 23, 25]:
    doc = corpus_df.iloc[i]
    print(f"\n[{doc['category']}] {doc['topic']}")
    print(f"  Content: {doc['content'][:120]}...")

## Step 2: Embed the Entire Corpus

We'll now convert each document into a numerical embedding. This is a one-time preprocessing step. In a production system, you would:
1. Embed all documents once
2. Store the embeddings in a vector database (e.g., FAISS, Pinecone, Weaviate)
3. Only embed new queries at search time

This makes search extremely fast, even for millions of documents!

In [ ]:
# Extract document content for embedding
corpus_texts = corpus_df["content"].tolist()

# Generate embeddings for all documents using the strict helper API
corpus_request = EmbeddingRequest(texts=corpus_texts)
corpus_response = generate_embeddings(corpus_request)
corpus_embeddings = corpus_response.vectors

print(f"\nCorpus embeddings shape: {corpus_embeddings.shape}")
print(f"  - {corpus_embeddings.shape[0]} documents")
print(f"  - {corpus_embeddings.shape[1]} dimensions per document")
print(f"Model used: {corpus_response.model_name}")

## Step 2.5: Visualize Document Relationships

Before we perform searches, let's visualize how our documents relate to each other in embedding space. We'll use UMAP to project the high-dimensional embeddings (384 dimensions) down to 2D while preserving their semantic relationships.

**What to Look For:**
- Documents from the same medical domain should cluster together
- Related topics across domains may show proximity (e.g., diabetes and insulin therapy)
- This visualization shows what the semantic search algorithm "sees" when comparing documents

In [ ]:
# Reduce corpus embeddings to 2D for visualization
reducer = umap.UMAP(
    n_components=2, random_state=42, n_neighbors=10, min_dist=0.3, metric="cosine"
)
corpus_embeddings_2d = reducer.fit_transform(corpus_embeddings)

print(f"Reduced corpus embeddings shape: {corpus_embeddings_2d.shape}")

# Create DataFrame with 2D coordinates for plotting
plot_df = corpus_df.copy()
plot_df["umap_1"] = corpus_embeddings_2d[:, 0]
plot_df["umap_2"] = corpus_embeddings_2d[:, 1]

# Create interactive scatter plot
fig = px.scatter(
    plot_df,
    x="umap_1",
    y="umap_2",
    color="category",
    text="topic",
    hover_data={"id": True, "topic": True, "category": True, "umap_1": False, "umap_2": False},
    labels={"umap_1": "UMAP Dimension 1", "umap_2": "UMAP Dimension 2", "category": "Medical Domain"},
    title="Document Corpus: Semantic Relationships in 2D Embedding Space",
    width=1200,
    height=800,
)

fig.update_traces(
    textposition="top center",
    marker={"size": 11, "line": {"width": 1.5, "color": "white"}},
    textfont={"size": 8},
)

fig.update_layout(
    font={"size": 11},
    title_font_size=16,
    showlegend=True,
    legend={
        "title": "Medical Domain",
        "orientation": "v",
        "yanchor": "top",
        "y": 1,
        "xanchor": "left",
        "x": 1.02,
    },
)

fig.show()

print("\n" + "=" * 80)
print("Observations:")
print("=" * 80)
print("• Documents from the same category cluster together spatially")
print("• Cardiovascular topics form a distinct region")
print("• Endocrine/metabolic documents group near each other")
print("• Mental health forms its own cluster")
print("• Cross-domain relationships appear (e.g., medications near related conditions)")
print("\nThis spatial organization is exactly what enables semantic search!")
print("When you search, we find documents closest to your query in this space.")

## Step 3: Perform a Semantic Search

Now comes the magic! We'll:
1. Take a user query
2. Embed the query using the same model
3. Calculate the similarity between the query and every document
4. Return the most similar documents

### How We Measure Similarity

We use **cosine similarity**, which measures the angle between two vectors:
- Cosine similarity = 1: Vectors point in the same direction (very similar)
- Cosine similarity = 0: Vectors are orthogonal (unrelated)
- Cosine similarity = -1: Vectors point in opposite directions (opposite meaning)

In practice, we often use **cosine distance** (1 - cosine similarity) where smaller values = more similar.

In [ ]:
def semantic_search(query: str, top_k: int = 3) -> pd.DataFrame:
    """Perform semantic search over the corpus using cosine similarity."""

    if top_k <= 0:
        raise ValueError("top_k must be a positive integer.")

    clean_query = query.strip()
    if not clean_query:
        raise ValueError("Query must contain at least one non-empty character.")

    # Step 1: Embed the query via the helper contract
    query_request = EmbeddingRequest(texts=[clean_query])
    query_embedding = generate_embeddings(query_request).vectors

    # Step 2: Calculate cosine distances between query and all documents
    distances = cdist(query_embedding, corpus_embeddings, metric="cosine")[0]

    # Step 3: Get indices of top-k most similar documents (smallest distances)
    valid_top_k = min(top_k, corpus_embeddings.shape[0])
    top_indices = np.argsort(distances)[:valid_top_k]

    # Step 4: Create results DataFrame with sequential ranks
    results = corpus_df.iloc[top_indices].copy().reset_index(drop=True)
    results["cosine_distance"] = distances[top_indices]
    results["similarity_score"] = 1 - distances[top_indices]
    results["rank"] = results.index + 1

    return results[["rank", "id", "topic", "content", "similarity_score"]]

## Example Queries

Let's test our semantic search engine with various queries!

In [ ]:
# Query 1: Heart attack symptoms
query1 = "What are the signs of a heart attack?"
print(f"Query: '{query1}'\n")
results1 = semantic_search(query1, top_k=3)
print("Top 3 Results:")
print("=" * 80)
for row in results1.itertuples(index=False):
    print(f"\n[Rank {row.rank}] Similarity: {row.similarity_score:.4f}")
    print(f"Topic: {row.topic}")
    print(f"Content: {row.content[:150]}...")
    print("-" * 80)

In [ ]:
# Query 2: Mental health (demonstrating new domain coverage)
query2 = "I'm feeling very sad and have no energy"
print(f"Query: '{query2}'\n")
results2 = semantic_search(query2, top_k=3)
print("Top 3 Results:")
print("=" * 80)
for row in results2.itertuples(index=False):
    print(f"\n[Rank {row.rank}] Similarity: {row.similarity_score:.4f}")
    print(f"Topic: {row.topic}")
    print(f"Content: {row.content[:150]}...")
    print("-" * 80)

In [ ]:
# Query 3: Medication information (cross-domain query)
query3 = "What blood thinners are available and how are they monitored?"
print(f"Query: '{query3}'\n")
results3 = semantic_search(query3, top_k=3)
print("Top 3 Results:")
print("=" * 80)
for row in results3.itertuples(index=False):
    print(f"\n[Rank {row.rank}] Similarity: {row.similarity_score:.4f}")
    print(f"Topic: {row.topic}")
    print(f"Content: {row.content[:150]}...")
    print("-" * 80)

## Try Your Own Query!

Modify the cell below to search for your own topics:

In [ ]:
# Your custom query here!
custom_query = "What should I know about seizures and epilepsy?"
print(f"Query: '{custom_query}'\n")
custom_results = semantic_search(custom_query, top_k=3)
print("Top 3 Results:")
print("=" * 80)
for row in custom_results.itertuples(index=False):
    print(f"\n[Rank {row.rank}] Similarity: {row.similarity_score:.4f}")
    print(f"Topic: {row.topic}")
    print(f"Content: {row.content[:150]}...")
    print("-" * 80)

## Analysis: How Does This Work?

### Key Observations from the Searches

1. **No Exact Matches Required:**
   - Query: "signs of a heart attack"
   - Top result: Document about "myocardial infarction symptoms"
   - The system understands these are the same concept even with different terminology!

2. **Semantic Understanding Across Terminology:**
   - Query: "high blood sugar"
   - Returns: Diabetes-related documents
   - The model knows hyperglycemia is related to diabetes and insulin

3. **Domain-Aware Ranking:**
   - Similarity scores help rank results by relevance
   - More relevant documents have higher scores (closer to 1.0)
   - Look back at the 2D visualization to see why certain documents rank higher!

### Connection to the Visualization

The 2D visualization you saw earlier shows the **landscape** that semantic search navigates:
- When you submit a query, it becomes a point in that same embedding space
- The search finds documents closest to your query point
- Documents that cluster together in the visualization will often co-appear in search results
- This is why cardiovascular queries return multiple cardiovascular documents - they're neighbors in embedding space

### Why This Matters for Digital Health

In healthcare, terminology varies widely across contexts:
- **Medical professionals:** "Myocardial infarction," "dyspnea," "hyperglycemia"
- **Patients:** "Heart attack," "shortness of breath," "high blood sugar"
- **Abbreviations:** "MI," "SOB," "HbA1c"
- **Drug names:** Generic vs. brand (e.g., "metformin" vs. specific brand names)

Semantic search handles all of these gracefully, making it ideal for:
- **Patient-facing search tools:** Patients can use everyday language
- **Clinical decision support:** Match symptoms to guidelines regardless of phrasing
- **Medical literature search:** Find relevant papers even with different terminology
- **EHR information retrieval:** Search notes using natural language
- **Cross-lingual applications:** Some embedding models support multiple languages

### The Power of Vector Similarity

The entire search process boils down to:
1. **One-time preprocessing:** Embed all documents (done once, stored in database)
2. **Query time:** Embed the query (fast, milliseconds)
3. **Similarity calculation:** Compare query embedding to document embeddings (optimized with FAISS for scale)
4. **Return top-k:** Sort by similarity and return best matches

This approach scales to millions of documents while maintaining sub-second response times!

## Extension: Scaling Up with FAISS (Optional)

Our current implementation uses **linear search**: we compare the query to every document. This is fine for 8 documents, but what about 8 million?

**FAISS (Facebook AI Similarity Search)** is a library for efficient similarity search at scale. It uses clever data structures to find approximate nearest neighbors in logarithmic time.

Here's a preview of how you would use FAISS (requires `pip install faiss-cpu`):

```python
import faiss

# Build FAISS index
dimension = corpus_embeddings.shape[1]  # 384
index = faiss.IndexFlatL2(dimension)  # L2 distance (similar to cosine for normalized vectors)
index.add(corpus_embeddings.astype('float32'))  # Add all corpus embeddings

# Search
query_request = EmbeddingRequest(texts=[query])
query_embedding = generate_embeddings(query_request).vectors.astype('float32')
distances, indices = index.search(query_embedding, k=3)  # Get top 3

# Results
top_docs = corpus_df.iloc[indices[0]]
```

FAISS can handle billions of embeddings efficiently, making it the go-to solution for production semantic search systems.

## Key Takeaways

1. **Semantic search understands meaning**, not just keywords
2. **Embeddings enable similarity calculations** via vector distance metrics
3. **Cosine similarity/distance** is the standard metric for comparing embeddings
4. **This approach scales** to millions of documents with proper indexing (FAISS)

### Next Steps

In future notebooks, we'll explore:
- **Retrieval-Augmented Generation (RAG):** Combining semantic search with LLMs
- **Fine-tuning embeddings:** Improving performance for domain-specific tasks
- **Multi-modal embeddings:** Searching images, audio, and text together

You now have the foundational knowledge to build intelligent search and retrieval systems for digital health applications!